# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #4 — The Freshness Multiplier

**Core claim:** Refreshing older pages yields large performance improvements (3.2x health boost, 57x impression boost for 365+ day content refreshed within 30 days).

**Methodology questions:**
1. How were pages selected for refresh — random assignment, editor judgment, or an existing performance-based rule? If chosen, what criteria drove the choice?
2. Is there a comparison group of unrefreshed pages matched on the same pre-refresh trajectory, or is "stale" content just whatever wasn't touched?
3. Does the 57x figure report an effect size with sample sizes for the refreshed vs. stale bucket, or could it come from a small, cherry-picked bucket?

**Why this matters (writing-honest-claims / selection bias check):** pages were not randomly assigned to be refreshed. Editors likely chose pages already showing signs of life (seasonal spikes, existing backlinks, internal priority). Without a control group, the reported lift is a mix of the refresh action and the choosing action — this is a selection-bias problem, not a leakage problem.

---

### Finding #2 — The Content Performance Curve

**Core claim:** Content follows a lifecycle peaking at 61-90 days (health score 33) and decaying steeply after 270 days (health score 14).

**Methodology questions:**
1. What features feed into Health Score, and do any of them overlap with the variables used to explain the curve (age, position, impressions)?
2. If Health Score were decomposed into its components, does the "peak at 61-90 days" pattern hold for each component separately, or does it disappear for some and dominate for others?
3. What's the base rate / naive comparison for the paper's claimed 71% holdout accuracy model — what would guessing the majority class alone score?

**Why this matters (hunting-leakage-and-validating / label-derived features):** Health Score is built from Impressions, Position, CTR, and Scroll Depth. In the paper's own appendix, Average Position (43%) and Impressions (32%) account for 75% of the score's predicted variance. Explaining a "lifecycle curve" of Health Score using age, position, and impressions is close to circular — it restates how the score is constructed rather than proving an independent lifecycle law.

In [1]:
# No computation needed for Section 1 — this is a methodology audit of the
# paper's claims, not a re-analysis of the paper's underlying data (which
# we don't have direct access to). See the markdown cell above for the
# two findings, their evidence, and the audit questions.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model (`w05_model.ipynb`) already used a grouped split by `client_id`, but reported precision 1.0, recall 0.999, and accuracy 1.0 — a near-perfect score that the leakage checklist flags as suspicious rather than something to celebrate. The top two coefficients, `impressions_last_30d` (-7.92) and `impressions_prev_30d` (+7.91), nearly cancel — a classic sign of features mechanically tied to how `trend_pct` (and therefore `is_declining_label`) was computed.

**The confession test** (train once with these suspect features, once without) confirms it: accuracy collapses from 1.0 to ~0.67 when the suspects are removed — right in line with the "collapse from ~1.0 to ~0.7 is the confession" pattern the skill describes. At a 62.8% base rate, the honest model shows roughly 4-5 points of real skill above guessing the majority class, not near-perfect prediction.

In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

RANDOM_SEED = 42

url = 'https://raw.githubusercontent.com/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Missingness handling: flags instead of blind fillna(0)
df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count'] = df['word_count'].fillna(df['word_count'].median())
df['is_declining_label'] = (df['trend_pct'] < 0).astype(int)

def run_model(data, feature_cols, seed=RANDOM_SEED, split_col='client_id'):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(data, groups=data[split_col]))
    train_df, test_df = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()
    X_train, y_train = train_df[feature_cols], train_df['is_declining_label']
    X_test, y_test = test_df[feature_cols], test_df['is_declining_label']
    imputer = SimpleImputer(strategy='median', add_indicator=True)
    X_train_i = imputer.fit_transform(X_train)
    X_test_i = imputer.transform(X_test)
    model = LogisticRegression(max_iter=5000, random_state=seed)
    model.fit(X_train_i, y_train)
    preds = model.predict(X_test_i)
    return {
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds),
        'accuracy': accuracy_score(y_test, preds),
        'base_rate': y_test.mean(),
    }

forbidden_always = ['is_declining_label', 'trend_direction', 'trend_pct', 'content_id', 'client_id', 'content_type']
suspects = ['impressions_last_30d', 'impressions_prev_30d']
all_numeric = [c for c in df.columns if c not in forbidden_always and pd.api.types.is_numeric_dtype(df[c])]

with_suspects = run_model(df, all_numeric)
without_suspects = run_model(df, [c for c in all_numeric if c not in suspects])

confession_table = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'Accuracy', 'Base Rate'],
    'WITH suspects (impressions_*_30d)': [round(with_suspects[k], 3) for k in ['precision','recall','accuracy','base_rate']],
    'WITHOUT suspects': [round(without_suspects[k], 3) for k in ['precision','recall','accuracy','base_rate']],
})
print("Leakage confession test — does the score collapse toward the base rate?\n")
print(confession_table.to_markdown(index=False))

Leakage confession test — does the score collapse toward the base rate?

| Metric    |   WITH suspects (impressions_*_30d) |   WITHOUT suspects |
|:----------|------------------------------------:|-------------------:|
| Precision |                               1     |              0.716 |
| Recall    |                               1     |              0.786 |
| Accuracy  |                               1     |              0.67  |
| Base Rate |                               0.628 |              0.628 |


**Note on "time-aware" split:** the starter CSV (`data/raw/content_refresh_anonymized.csv`) does not include a `report_date`-style column — only `days_since_last_update`, which is a duration, not a calendar date. Sorting by it and splitting 80/20 would split by freshness tier, not by past-vs-future, so it would be misleading to label that a genuine time split. A real time-aware honest split requires the warehouse release tables (`fact_content_daily_performance`, keyed by `report_date`), which is out of scope for this notebook. The grouped split by `client_id` (shown above, "WITHOUT suspects" column) is the honest number I'm reporting: **~67% accuracy against a 62.8% base rate.**

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the seven-item attack checklist against the cleaned feature set (impressions siblings removed):

| # | Checklist item | Status |
|---|---|---|
| 1 | Timeline drawn: features strictly before label window | ⚠️ Partial — `clicks_last_30d`/`sessions_last_30d` share the same 30-day window structure as the removed impressions pair. Their correlation with the label is weak (0.006–0.053), so they're not dominating like the impressions pair did, but they're the same category of risk and worth naming rather than assuming safe. |
| 2 | No label-derived or sibling columns in features | ✅ Confirmed by the confession test — accuracy collapsed 1.0 → ~0.67 when `impressions_last_30d`/`impressions_prev_30d` were removed. |
| 3 | No product flags / existing-system scores as features | ✅ Clean — no precomputed decision flag or score in the feature set. |
| 4 | Split grouped by repeating entity | ✅ `GroupShuffleSplit` on `client_id`. |
| 5 | Base rate printed next to every metric | ✅ 62.8% reported throughout. |
| 6 | Top feature importance sanity-checked | ✅ Passes now — top coefficient on the cleaned set is only ~-0.11 (`clicks_last_30d`), a healthy spread instead of one dominant feature. |
| 7 | Metrics recomputed out-of-fold | ⚠️ Not done — current numbers are a single train/test holdout, not cross-validated. Worth naming as a limitation. |

**Two data-quality issues found and fixed along the way (not leakage, but worth documenting):**
- `avg_position == 0` (1,205 rows) was being fed to the model as a literal top rank, when the data dictionary states 0 means "no data." Fixed by flagging these as missing and imputing from valid-only values.
- `age_tier_order` correlates 0.951 with `content_age_days` — a near-duplicate binned version of the same signal, kept in for now but flagged as redundant.

In [3]:
# Re-run cleaned model with the avg_position fix applied, and print the
# coefficient table + before/after effect of the fix.

df2 = pd.read_csv(url)
df2['has_word_count'] = df2['word_count'].notna().astype(int)
df2['word_count'] = df2['word_count'].fillna(df2['word_count'].median())
df2['is_declining_label'] = (df2['trend_pct'] < 0).astype(int)

# FIX: avg_position == 0 means "no data", not rank zero
df2['has_avg_position'] = (df2['avg_position'] != 0).astype(int)
df2.loc[df2['avg_position'] == 0, 'avg_position'] = np.nan
df2['avg_position'] = df2['avg_position'].fillna(df2['avg_position'].median())

features_clean = [c for c in all_numeric if c not in suspects] + ['has_avg_position']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df2, groups=df2['client_id']))
train_df, test_df = df2.iloc[train_idx].copy(), df2.iloc[test_idx].copy()
X_train, y_train = train_df[features_clean], train_df['is_declining_label']
X_test, y_test = test_df[features_clean], test_df['is_declining_label']

imputer = SimpleImputer(strategy='median', add_indicator=True)
X_train_i = imputer.fit_transform(X_train)
X_test_i = imputer.transform(X_test)
model = LogisticRegression(max_iter=5000, random_state=RANDOM_SEED)
model.fit(X_train_i, y_train)
preds = model.predict(X_test_i)

coefs = pd.DataFrame({
    'Feature': imputer.get_feature_names_out(X_train.columns),
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(f"Precision: {precision_score(y_test, preds, zero_division=0):.3f} | "
      f"Recall: {recall_score(y_test, preds):.3f} | "
      f"Accuracy: {accuracy_score(y_test, preds):.3f} | "
      f"Base Rate: {y_test.mean():.3f}\n")
print("Top 8 coefficients after the avg_position fix:\n")
print(coefs.head(8).to_markdown(index=False))

Precision: 0.714 | Recall: 0.798 | Accuracy: 0.672 | Base Rate: 0.628

Top 8 coefficients after the avg_position fix:

| Feature              |   Coefficient |
|:---------------------|--------------:|
| clicks_last_30d      |    -0.119246  |
| age_tier_order       |    -0.0432506 |
| engaged_sessions_90d |    -0.0427143 |
| clicks_90d           |     0.0377723 |
| has_avg_position     |     0.031494  |
| ctr                  |    -0.0289345 |
| days_with_sessions   |    -0.0275662 |
| scroll_events_90d    |     0.0273595 |


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (bold) claim:** "The model predicts declining content with 81.7% precision."

**Rewritten:** Under a grouped validation split on held-out test data, the model **measured** roughly 71-72% precision and ~67% accuracy for flagging declining content — modestly above the 62.8% base rate. An earlier version of this model reported near-perfect scores (precision 1.0, accuracy 1.0), which turned out to be an artifact of two label-sibling features (`impressions_last_30d`, `impressions_prev_30d`) leaking information about how the label itself was computed. Once those features were removed, the honest, **decision-support** reading is: this model provides a modest signal for prioritizing which content to review for decline, not a near-certain prediction.

---

### Paper claim rewrites (Findings #4 and #2)

**Finding #4 — Freshness Multiplier:**
> In this dataset, pages aged 365+ days that had been refreshed within the last 30 days were **observed** with a substantially higher health score and impression count than stale pages of the same age (3.2x and 57x respectively). Because pages were not randomly assigned to be refreshed, this comparison does not isolate the effect of refreshing from the effect of which pages editors chose to refresh. The honest form is decision-support: "aged pages that get refreshed tend to look healthier afterward in this portfolio," not "refreshing an aged page will produce this lift."

**Finding #2 — Content Performance Curve:**
> Health Score, as **measured** in this dataset, is highest for content aged 61-90 days (33) and lowest for content aged 271-365 days (14). Because Health Score is calculated from Impressions, Position, CTR, and Scroll Depth — the same categories of signal used elsewhere to explain performance — this "lifecycle curve" is partly a restatement of how the score is built, not an independently validated pattern. The honest reading: "these score components tend to be highest around 61-90 days in this portfolio," not evidence of a universal content lifecycle law.

In [4]:
# No computation needed for Section 4 — this is a language/claims audit.
# See the markdown cell above for the rewritten claims.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.